# E91 Quantum Key Distribution

This notebook demonstrates the E91 Quantum Key Distribution protocol from Chapter 10, Key Applications of Entanglement.

[![Open In Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Polaris-QCI/Quantum-Computing-and-Information-ed2/main?labpath=Qiskit%2F10-Key-Applications-of-Entanglement%2Fe91-quantum-key-distribution-demo.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Polaris-QCI/Quantum-Computing-and-Information-ed2/blob/main/Qiskit/10-Key-Applications-of-Entanglement/e91-quantum-key-distribution-demo.ipynb)

## 0. Setup

The cells below install the packages and import the libraries needed by this demo.

In [1]:
#@title Install Qiskit and other dependencies
%pip install pylatexenc qiskit qiskit-aer qiskit-ibm-runtime

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 18.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 26.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 26.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 28.8 MB/s  0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.7 MB/s  0:00:00
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136897 sha256=d62ec45108cce7456f4ca2787f889e5154634aaa3692f4018fb2336e50bf3225
  Stored in directory: /Users/jakeschaefer/Library/Caches/

In [1]:
# Imports
import jax
import jax.random as jrnd
import random
from IPython.display import display, Markdown
from math import cos, sin, pi as π
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit import ClassicalRegister, QuantumCircuit, QuantumRegister
from qiskit.primitives.containers.data_bin import DataBin
from qiskit.primitives.containers.primitive_result import PrimitiveResult
from qiskit.primitives.containers.sampler_pub_result import SamplerPubResult
from qiskit.primitives.containers.bit_array import BitArray


## 1. E91 bit

The next cell implements the E91 protocol for a single potential key bit. 

In [18]:
from qiskit.circuit.instructionset import InstructionSet



def get_res(name: str, res: PrimitiveResult):
  res0: SamplerPubResult = res[0]
  _bin: DataBin = res0.data
  _name_bin: BitArray = _bin[name]
  return next(iter(_name_bin.get_counts().keys()))

def e91_bit(key):
  q = [q0, q1] = QuantumRegister(2)
  c = ClassicalRegister(2, 'c')
  qc = QuantumCircuit(q, c)

  # Generate the Ψ⁻ Bell state
  # out: InstructionSet = 
  qc.x(q0)
  qc.x(q1)
  qc.h(q0)
  qc.cx(q0, q1)

  # Alice's basis choices
  # a1 = Z (no rotation needed)
  # a2 = W = 1/√2(Z+X)
  # a3 = X
  k1, k2, key = jrnd.split(key, 3)
  
  def z_basis(q, i):
    return i + 1

  def w_basis(q, i):
    qc.ry(-π/4, q)
    return i + 1
  
  def x_basis(q, i):
    qc.h(q)
    return i + 1

  # Bob's basis choices
  # b1 = W = 1/√2(Z+X)
  # b2 = X
  # b3 = V = 1/√2(Z-X)
  
  def v_basis(q, i):
    qc.ry(π/4, q)
    return i + 1
  
  with jax.disable_jit():
    idx = jrnd.randint(k1, (), 0, 3)
    alice_basis_num = jax.lax.switch(
      idx, 
      (
        z_basis, # Z basis
        w_basis, # W basis
        x_basis, # X basis
      ),
      q0, idx
    )
    idx = jrnd.randint(k2, (), 0, 3)
    bob_basis_num = jax.lax.switch(
      idx,
      (
        w_basis, # W basis
        x_basis, # X basis
        v_basis, # V basis
      ),
      q1,
      idx
    )

  qc.measure(q, c)

  simulator = AerSimulator()
  sampler = Sampler(simulator)
  job = sampler.run([qc], shots=1)
  job_res = job.result()
  res = get_res('c', job_res)
  return {
    "alice_basis": ('a' + str(alice_basis_num)),
    "bob_basis": ('b' + str(bob_basis_num)),
    "alice_result": res[0],
    "bob_result": res[1],
    'key': key
  }


In [19]:
cols = ['#', 'Alice Basis', 'Bob Basis', 'Use for Key', 'Measurement Result', 'Use for Checking']
output = '| ' + ' | '.join(cols) + ' |\n'
output += '|' + '|'.join(map(lambda x: '-'*(len(x) + 2), cols)) + '|\n'

key = jrnd.PRNGKey(0)
for i in range(20):
  bit = e91_bit(key)
  key = bit['key']
  alice_basis = bit['alice_basis']
  bob_basis = bit['bob_basis']
  use_for_key = "✓" if alice_basis == "a2" and bob_basis == "b1" or alice_basis == "a3" and bob_basis == "b2" else " "
  measurement_result = f"**{bit['alice_result']}**" if use_for_key == "✓" else bit['alice_result']
  use_for_checking = "✓" if alice_basis != "a2" and bob_basis != "b2" else "✗"
  output += f"| {i+1} | {alice_basis} | {bob_basis} | {use_for_key} | {measurement_result} | {use_for_checking} |\n"
display(Markdown(output))

| # | Alice Basis | Bob Basis | Use for Key | Measurement Result | Use for Checking |
|---|-------------|-----------|-------------|--------------------|------------------|
| 1 | a2 | b3 |   | 0 | ✗ |
| 2 | a2 | b1 | ✓ | **1** | ✗ |
| 3 | a3 | b3 |   | 1 | ✓ |
| 4 | a2 | b1 | ✓ | **0** | ✗ |
| 5 | a1 | b1 |   | 0 | ✓ |
| 6 | a3 | b3 |   | 1 | ✓ |
| 7 | a1 | b3 |   | 1 | ✓ |
| 8 | a1 | b2 |   | 0 | ✗ |
| 9 | a2 | b2 |   | 1 | ✗ |
| 10 | a2 | b1 | ✓ | **0** | ✗ |
| 11 | a1 | b1 |   | 0 | ✓ |
| 12 | a1 | b3 |   | 1 | ✓ |
| 13 | a3 | b2 | ✓ | **1** | ✗ |
| 14 | a1 | b1 |   | 0 | ✓ |
| 15 | a1 | b3 |   | 1 | ✓ |
| 16 | a1 | b2 |   | 0 | ✗ |
| 17 | a1 | b1 |   | 0 | ✓ |
| 18 | a1 | b1 |   | 1 | ✓ |
| 19 | a3 | b2 | ✓ | **0** | ✗ |
| 20 | a3 | b1 |   | 1 | ✓ |


Overall, I dont really like how they define operations on a quantum circuit as in-place. For both of these, I would probably edit them by adding Bloch-sphere plots, as these tend to be helpful.
<!-- It would be much better as a functional, scanned over system I think -->

### E91 protocol.


This somewhat helped me conceptually: it helped by explicitly showing the creation of the Bell pair, rotating into chosen bases, measuring both halves, and classifying each round as key-generation or type checking depending on the basis choices. One thing I was anonoyed with was that it did not compute the correlators $E(a_i, b_j)$ or CHSH parameter $S$. I therefore couldn't clearly see the connection to Bell's inequality or CHSH. To modify it, I would fix this.


### BB84 protocol

I especially liked the convergence graph in this one. Here, the steps of building a QRNG from a Hadamard gate plus a measurement and using that randomness to preparation/measurement behavior, i.e., Alice encoding a bit into one of two incompatible bases and Bob measuring in a randomly chosen basis and then printing the table of Alice and Bob basis outcomes was insightful. In particular, seeing the results in a tabular form allowd me to looka t all the different possible cases. I think it could've been more enlightening in showing the no-cloning theorem.






### E91 protocol.

- how this relates to previous understandings of CHSH inequality, and the BB84 and E91 protocols?
- did it help, confuse you more, not help much ? How ?
- How would you modify this to make it more interesting or compelling?
- 

- Alice and Bob establish a shared key by separately performing measurements via randomly chosen bases on sets of EPR pairs. 
- Then, compare bases and only keep those where they agree as KEY

- randomly chooses A, B measurement settings, applies basis rotations, takes one-shot measurements, and labels which rounds are for key-gen vs which are for checking
- alic is not preparing a single-qubit state for bob, but instead both parties measure halves of an entangled pair. 
- measurement bases are chosen from Bell-test geometry -> E91 is really "QKD built on Bell/CHSH structure"


- here, Alice anc Bob do not need Alice to actively prepare a qubit and send it to Bob during the key step, as in BB84. Instead:
  - they can start with entangled pairs already distributed between them
  - later, each side just measures their own half
- protocol only needs them to share entanglement, not for Alice to "send the key state live" during the measurement phase

caveat:
- the source does not have to be trusted in the ideal story
- security comes from the observed correlations, especially Bell-inequality violation

classical channel carries:
- which measurement basis each party used
- which rounds are kept for key generation
- which rounds are sacrificed for bell testing / error estimation




### BB84 protocol


- QRNG built from Hadamard plus measurement, and uses that rqandomness to choose preparation/measurement behavior, printing a table of A, B basis choices and outcomes.
- performs key sifting, adds optional Eve, and finally estimates Eve-detection prob as number of bits grows,
- incompatible bases plus no-cloning makes eavesdropping detectable. 

Context:
- alice wants to send bob a sectret random key
- eve may listen to quantum channel and public classical channel (?)
- alice and bob trust an authenticated public classical channel. Eve can hear it but cannot fake being alice or bob.
- basis mismatch does not mean there is an eavesdropper.
- signal for eavesdropping: when Alice and Bob use same basis, but their bit values disagree too often.



Setup
1. Alice creates a random sequence of 1s and 0s
2. Alice encodes each bit of this sequence into polarization of photon
3. Encodes this into one of two bases chosen randomly
4. Alice sends qubits to Bob along/via quantum channel
5. Bob measures polarization randomly in 'x' (D/A) or 'z' (H/V)
6. Alice calls bob on classical open channel at all transmitted photons
7. compare when choices were x or z [encoding + decoding]
8. polarizations are correlated for same choice (1/2 of the time)
9. discard different choices
10. now, classically verify if indeed correlated (eavesdropper)
11. 
  

- every round, alice randomly chooses a bit 0 or 1 and a basis HV or DA. She prepares a qubit in one of four states: H/V or D/A and she sends that qubit to Bob.
  - alice knows the basis she used and the bit she encoded
- bob does not know alice's basis. He randomly chooses his own basis HV or DA and measures. 
  - If he happened to choose the same basis as Alice, he should get the correct bit. 
  - If he chose the wrong basis, his result is effectively random.
- Bob then knows his own basis, his own measurement result, not alice's basis until later

- What eve knows/does:
  - intecept the qubit
  - choose a random basis
  - measure
  - resend a new qubit based on her result
- Eve knows:
  - the protocol
  - all later public discussion
  - her own chosen basis and result
- why same-basis errors reveal Eve:
  - sps alice sent in HV
  - if eve also measures in HV, she learns the bit without disturbance
  - if eve measures in DA, she disturbs the state, so even if bob later uses HV, his result can be wrong.


- no cloning thm: an unknown state cannot be reproduced

- bloch sphere visual would improve both